# Test Cow Ear Tag Detector YOLO26n 🐮

This notebook evaluates the [Cow Ear Tag Detector YOLO26n 🐮](https://huggingface.co/mirandamurphy/cow-ear-tag-detector-yolo26n), a YOLO26n model fine-tuned to detect ear tags in cows.

**Notebook Overview**
- Loads the prepared YOLO-format dataset
- Loads the fine-tuned YOLO26n model
- Evaluates the model on the held-out test set
- Logs evaluation metrics

**Base Model:** [Ultralytics YOLO26n](https://arxiv.org/abs/2606.03748) \
**Trained Model:** [Cow Ear Tag Detector YOLO26n 🐮](https://huggingface.co/mirandamurphy/cow-ear-tag-detector-yolo26n)

**Dataset Source:** CEID-D (Cow Ear Tag Detection Dataset), available on [Kaggle](https://www.kaggle.com/datasets/fandaoerji/cow-eartag-detection-dataset/data).

**Author:** Miranda Murphy \
**Contact:** mirandamurphy.dev@protonmail.com \
**License:** AGPL-3.0

For full dataset preparation details, model training hyperparameters, licensing, and model information see the repository README and [model card](https://huggingface.co/mirandamurphy/cow-ear-tag-detector-yolo26n).


In [ ]:
%pip install -q dagshub mlflow ultralytics

import os
from pathlib import Path

import dagshub
import mlflow
import mlflow.artifacts
from ultralytics import settings, YOLO
from google.colab import userdata

In [ ]:
# Load repo credentials from Colab Secrets
DAGSHUB_USER = userdata.get("DAGSHUB_USER")
DAGSHUB_REPO = userdata.get("DAGSHUB_REPO")
DAGSHUB_TOKEN = userdata.get("DAGSHUB_TOKEN")

REMOTE_PATH = userdata.get("REMOTE_PATH")
LOCAL_PATH = userdata.get("LOCAL_PATH")

os.environ["DAGSHUB_USER"] = DAGSHUB_USER
os.environ["DAGSHUB_REPO"] = DAGSHUB_REPO
os.environ["DAGSHUB_TOKEN"] = DAGSHUB_TOKEN

os.environ["REMOTE_PATH"] = REMOTE_PATH
os.environ["LOCAL_PATH"] = LOCAL_PATH

In [ ]:
# Download preprocessed dataset into the Colab VM
!dagshub download --bucket "$DAGSHUB_USER/$DAGSHUB_REPO" "$REMOTE_PATH" "$LOCAL_PATH"

In [ ]:
DATASET_DIR = Path("/content/datasets/yolo")
DATA_YAML = DATASET_DIR / "data.yaml"

print(f"Dataset path: {DATASET_DIR}")
print(f"data.yaml exists: {DATA_YAML.exists()}")

image_dir = DATASET_DIR / "images" / "test"

if image_dir.exists():
    count = len(list(image_dir.glob("*")))
    print(f"Images in test split: {count}")
else:
    print("Test split not found")

In [ ]:
# Setup MLflow tracking via DagsHub
EXPERIMENT_NAME = userdata.get("EXPERIMENT_NAME")
RUN_NAME = userdata.get("RUN_NAME")
RUN_ID = userdata.get("RUN_ID")
ARTIFACT_PATH = userdata.get("ARTIFACT_PATH")
DST_PATH = userdata.get("DST_PATH")

os.environ["MLFLOW_EXPERIMENT_NAME"] = EXPERIMENT_NAME
os.environ["MLFLOW_RUN"] = RUN_NAME
os.environ["RUN_ID"] = RUN_ID
os.environ["ARTIFACT_PATH"] = ARTIFACT_PATH
os.environ["DST_PATH"] = DST_PATH
os.environ["MLFLOW_KEEP_RUN_ACTIVE"] = "True"

settings.update({"mlflow": True})

dagshub.init(repo_owner=DAGSHUB_USER,
             repo_name=DAGSHUB_REPO,
             mlflow=True
             )

mlflow.set_experiment(EXPERIMENT_NAME)

print("MLflow tracking URI: ", mlflow.get_tracking_uri())

In [ ]:
# Download the trained model from the specified MLflow run.
best_model_path = mlflow.artifacts.download_artifacts(
    run_id=RUN_ID,
    artifact_path=ARTIFACT_PATH,
    dst_path=DST_PATH
)

best_model = YOLO(best_model_path)

# Evaluate on the labeled test split
metrics = best_model.val(
    data=str(DATA_YAML),
    split='test')

val_metrics = {
    "test_mean_precision": metrics.box.mp,
    "test_mean_recall": metrics.box.mr,
    "test_fitness": metrics.box.fitness(),
    "test_map50-95": metrics.box.map,
    "test_map50": metrics.box.map50,
    "test_map75": metrics.box.map75,
    "test_ms_image_preprocess": metrics.speed["preprocess"],
    "test_ms_image_inference": metrics.speed["inference"],
    "test_ms_image_loss": metrics.speed["loss"],
    "test_ms_image_postprocess": metrics.speed["postprocess"],
}

mlflow.log_metrics(val_metrics)

In [ ]:
run = mlflow.active_run()

if run:
    mlflow.end_run()
else:
    print("No active MLflow run found.")